In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
q12025 = pd.read_excel('2025comparendosQ1.xlsx', header=9)
q22025 = pd.read_excel('2025comparendosQ2.xlsx', header=9)
q32025 = pd.read_excel('2025comparendosQ3.xlsx', header=9)
q42025 = pd.read_excel('2025comparendosQ4.xlsx', header=9)

In [ ]:
q12025['TRIMESTRE'] = 1
q22025['TRIMESTRE'] = 2
q32025['TRIMESTRE'] = 3
q42025['TRIMESTRE'] = 4

In [ ]:
totalcomparendos2025 = (
    q12025['CANTIDAD'].sum() +
    q22025['CANTIDAD'].sum() +
    q32025['CANTIDAD'].sum() +
    q42025['CANTIDAD'].sum()
)
print(totalcomparendos2025)

In [ ]:
df2025total = pd.concat([q12025, q22025, q32025, q42025], ignore_index=True)

In [ ]:
filepath = 'PPED-AreaMun-2018-2042VP (1).xlsx'

df = pd.read_excel(filepath, sheet_name='PobMunicipalxÁrea', skiprows=7)
df = df.dropna(subset=['AÑO', 'ÁREA GEOGRÁFICA'])

df2025pob = df[(df['AÑO'] == 2025) & (df['ÁREA GEOGRÁFICA'] == 'Total')].copy()
dfresultado = df2025pob[['DP', 'DPNOM', 'MPIO', 'DPMP', 'TOTAL']]
dfresultado.columns = ['CódigoDepartamento', 'Departamento', 'CÓDIGO DIVIPOLA', 'MUNICIPIO', 'PoblacionTotal2025']
dfresultado = dfresultado.reset_index(drop=True)

In [ ]:
dfmerge = dfresultado[['CÓDIGO DIVIPOLA', 'Departamento', 'MUNICIPIO', 'PoblacionTotal2025']].copy()

dfmerge['CÓDIGO DIVIPOLA'] = pd.to_numeric(dfmerge['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')
df2025total['CÓDIGO DIVIPOLA'] = pd.to_numeric(df2025total['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')

df2025total = df2025total.merge(dfmerge, on='CÓDIGO DIVIPOLA', how='left', suffixes=('', '_pob'))

df2025total['DEPARTAMENTO'] = df2025total['Departamento'].fillna(df2025total['DEPARTAMENTO'])
df2025total['MUNICIPIO'] = df2025total['MUNICIPIO_pob'].fillna(df2025total['MUNICIPIO'])

df2025total = df2025total.rename(columns={'PoblacionTotal2025': 'PoblacionTotalMunicipio2025'})
df2025total = df2025total.drop(columns=['Departamento', 'MUNICIPIO_pob'])

In [ ]:
municipio2025 = (
    df2025total
    .groupby(['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO'], as_index=False)
    .agg({'comparendos': ('CANTIDAD', 'sum'), 'poblacion': ('PoblacionTotalMunicipio2025', 'max')})
)

municipio2025['poblacion'] = municipio2025['poblacion'].replace(0, pd.NA)
municipio2025['tasacomparendos100k'] = municipio2025['comparendos'] / municipio2025['poblacion'] * 100000

totalcomparendos_calc = municipio2025['comparendos'].sum()
print(f'Total: {totalcomparendos_calc}')
print(f'Coincide: {totalcomparendos_calc == totalcomparendos2025}')

print(municipio2025.sort_values('tasacomparendos100k', ascending=False).head(10))

In [ ]:
df2025total = df2025total.merge(
    municipio2025[['CÓDIGO DIVIPOLA', 'tasacomparendos100k']],
    on='CÓDIGO DIVIPOLA',
    how='left'
)

In [ ]:
def clasificar_categoria(pob):
    if pob > 500001:
        return 'Categoría especial'
    elif 100001 < pob <= 500000:
        return 'Categoría 1'
    elif 50001 < pob <= 100000:
        return 'Categoría 2'
    elif 30001 < pob <= 50000:
        return 'Categoría 3'
    elif 20001 < pob <= 30000:
        return 'Categoría 4'
    elif 10001 < pob <= 20000:
        return 'Categoría 5'
    else:
        return 'Categoría 6'

df2025total['categoria_municipio'] = df2025total['PoblacionTotalMunicipio2025'].apply(clasificar_categoria)

In [ ]:
df2025total.to_csv('Comparendos2025conPoblacionyTasa.csv', index=False)